In [ ]:
import glob
import os

import numpy as np
import rasterio
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize

# Visualize all spectral bands for a single randomly drawn pre-training sample.
# Rows = bands, Columns = time steps.

BANDS = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
N_TIMESTEPS = 5
SEED = None  # set an int for a reproducible draw

_s2_data_dir = os.environ.get(
    "S2_DATA_DIR",
    "../../../sentinel/data",
)
output_dir = os.path.join(_s2_data_dir, "SSL-consecutive")

In [ ]:
# Draw one sample directory via reservoir sampling (avoids listing all dirs upfront)
rng = np.random.default_rng(SEED)

chosen = None
seen = 0
with os.scandir(output_dir) as it:
    for entry in it:
        if not entry.is_dir():
            continue
        seen += 1
        if chosen is None or rng.integers(0, seen) == 0:
            chosen = entry.path

if chosen is None:
    raise FileNotFoundError(f"No sample directories found under: {output_dir}")

print(f"Selected sample: {os.path.basename(chosen)}  (from {seen} total)")

timestep_dirs = sorted(
    d for d in glob.glob(os.path.join(chosen, "*")) if os.path.isdir(d)
)
print(f"Timesteps found: {len(timestep_dirs)}")

In [ ]:
def load_bands(tif_path: str, n_bands: int) -> np.ndarray:
    """Return (n_bands, H, W) float32 array; NaN preserved for gaps."""
    with rasterio.open(tif_path) as src:
        data = src.read(list(range(1, n_bands + 1))).astype(np.float32)
    return data


def stretch(img: np.ndarray, pmin: float = 2, pmax: float = 98) -> np.ndarray:
    """Robust per-image percentile stretch to [0, 1]; gaps stay as-is."""
    lo = np.nanpercentile(img, pmin)
    hi = np.nanpercentile(img, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 10000.0
    return np.clip((img - lo) / (hi - lo + 1e-6), 0, 1)


# Load all timesteps: list of (n_bands, H, W) arrays
all_data = []
timestamps = []
for t, tdir in enumerate(timestep_dirs[:N_TIMESTEPS]):
    ts_name = os.path.basename(tdir)[:8]
    timestamps.append(f"{ts_name[:4]}/{ts_name[4:6]}/{ts_name[6:8]}")
    tif = os.path.join(tdir, "S2C.tif")
    arr = load_bands(tif, len(BANDS))
    all_data.append(arr)

n_t = len(all_data)
n_b = len(BANDS)
TARGET_T = n_t // 2
print(f"Loaded {n_t} timesteps, {n_b} bands, shape per timestep: {all_data[0].shape}")

In [ ]:
# Compute per-band global normalization range across all timesteps
# so each row has a consistent colour scale.
band_lo = np.zeros(n_b)
band_hi = np.ones(n_b)
for b in range(n_b):
    vals = np.concatenate([arr[b].ravel() for arr in all_data])
    lo = np.nanpercentile(vals, 2)
    hi = np.nanpercentile(vals, 98)
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        band_lo[b], band_hi[b] = lo, hi
    else:
        band_lo[b], band_hi[b] = 0.0, 10000.0

fig, axes = plt.subplots(
    n_b, n_t, figsize=(3.5 * n_t, 3 * n_b),
    squeeze=False,
)

for b, band_name in enumerate(BANDS):
    norm = Normalize(vmin=band_lo[b], vmax=band_hi[b])
    for t in range(n_t):
        ax = axes[b][t]
        img = all_data[t][b]  # (H, W)
        gap_mask = np.isnan(img)

        display = np.clip((img - band_lo[b]) / (band_hi[b] - band_lo[b] + 1e-6), 0, 1)
        display[gap_mask] = 0.0  # render gaps as black

        ax.imshow(display, cmap="gray", vmin=0, vmax=1, interpolation="nearest")

        gap_pct = 100.0 * gap_mask.sum() / gap_mask.size
        title = f"{timestamps[t]}"
        if t == TARGET_T:
            title += " (target)"
        if gap_pct > 0:
            title += f"\ngaps={gap_pct:.1f}%"
        ax.set_title(title, fontsize=8)
        ax.axis("off")

        if t == 0:
            ax.set_ylabel(band_name, fontsize=10, rotation=0, labelpad=35, va="center")
            ax.yaxis.set_label_coords(-0.15, 0.5)
            ax.axis("on")
            ax.set_xticks([])
            ax.set_yticks([])
            for spine in ax.spines.values():
                spine.set_visible(False)

sample_id = os.path.basename(chosen)
fig.suptitle(f"All bands — sample {sample_id}", fontsize=13, y=1.002)
plt.tight_layout()
plt.show()